# 🤖 The pi Agent Framework — `@earendil-works/pi-agent-core`

The capstone of the series. In **notebook 040** you wrote the tool-calling loop *by hand*:
send the context, look for `toolCall` blocks, execute them, append `toolResult`
messages, and call the model again — repeating until the model stops.

The **`Agent`** class does all of that for you. It owns the transcript, runs the
loop, executes your tools, and emits lifecycle **events** you can subscribe to for
live UIs. You bring the model, a system prompt, and a set of tools.

Because our model lives in a *custom* Azure `Models` collection (not one of pi's
built-in providers), we hand the `Agent` a small **`streamFn`** that delegates to
`models.streamSimple` — that's the only wiring needed.

> **Kernel:** Deno (pi.dev is ESM-only TypeScript; the Deno kernel runs it natively).

## 1. Load env & register Azure OpenAI

Same opening as every notebook in the series: load the `AZURE_PI_TEST_*` keys, then
register the Azure OpenAI provider via the shared `registerAzure()` helper.

In [ ]:
import { loadEnvUp } from "playground/env";
const env = await loadEnvUp();

In [ ]:
import { registerAzure } from "playground/azure";
const { models, model } = registerAzure();

## 2. Wire the `Agent` to our Azure model

`Agent` is model-agnostic: it drives whatever `streamFn` you give it. The signature is
`(model, context, options?) => AssistantMessageEventStream | Promise<...>` — exactly what
`models.streamSimple` provides. So the bridge is a one-liner.

In [ ]:
import { Agent, type StreamFn } from "@earendil-works/pi-agent-core";
import { Type } from "@earendil-works/pi-ai";

// The Agent calls this whenever it needs a model turn. We just forward to the
// Azure provider registered above.
const streamFn: StreamFn = (m, ctx, opts) => models.streamSimple(m, ctx, opts);

console.log("streamFn wired to azure-openai ✔");

## 3. Define tools as `AgentTool`s

An `AgentTool` is a `Tool` (name + description + TypeBox `parameters`) **plus**:
- a `label` for UI display, and
- an `execute(toolCallId, params, signal, onUpdate)` function that the Agent calls
  automatically when the model requests the tool. It returns
  `{ content: [...], details }`. (Throw on failure — don't hand-encode errors.)

We reuse the two tools from notebook 040, now as self-executing `AgentTool`s.

In [ ]:
import type { AgentTool } from "@earendil-works/pi-agent-core";

const calculatorTool: AgentTool = {
  name: "calculator",
  label: "Calculator",
  description: "Do arithmetic on two numbers. op is one of: add, sub, mul, div.",
  parameters: Type.Object({
    a: Type.Number({ description: "First operand" }),
    b: Type.Number({ description: "Second operand" }),
    op: Type.String({ description: "add | sub | mul | div" }),
  }),
  // The Agent validates args against the schema before calling execute.
  execute: (_toolCallId, params) => {
    const { a, b, op } = params as { a: number; b: number; op: string };
    const result =
      op === "add" ? a + b :
      op === "sub" ? a - b :
      op === "mul" ? a * b :
      op === "div" ? a / b :
      NaN;
    return Promise.resolve({
      content: [{ type: "text", text: String(result) }],
      details: { a, b, op, result },
    });
  },
};

const weatherTool: AgentTool = {
  name: "get_weather",
  label: "Get Weather",
  description: "Get the current weather for a city.",
  parameters: Type.Object({
    city: Type.String({ description: "City name" }),
  }),
  execute: (_toolCallId, params) => {
    const { city } = params as { city: string };
    // Canned, deterministic "forecast" so the notebook is reproducible offline.
    const text = `It is 22°C and sunny in ${city}, with a light breeze.`;
    return Promise.resolve({
      content: [{ type: "text", text }],
      details: { city, tempC: 22, conditions: "sunny" },
    });
  },
};

console.log("Defined tools:", [calculatorTool.name, weatherTool.name].join(", "));

## 4. Construct the `Agent` and subscribe to events

We pass `streamFn` plus an `initialState` (model, system prompt, tools). Then we
`subscribe()` to the agent's lifecycle events so we can watch it work:

| Event | Meaning |
|-------|---------|
| `message_update` | a streaming delta (text/thinking/tool-call) for the current assistant message |
| `tool_execution_start` | the agent is about to run one of your tools |
| `turn_end` | the assistant message + any tool results for that turn are final |
| `agent_end` | the run is complete; carries the full transcript |

In [ ]:
const agent = new Agent({
  streamFn,
  initialState: {
    model,
    systemPrompt:
      "You are a concise assistant. Use the provided tools when they help, " +
      "then give a short final answer.",
    tools: [calculatorTool, weatherTool],
  },
});

// Live observability: stream assistant text as it arrives, and log tool activity.
const unsubscribe = agent.subscribe((event) => {
  switch (event.type) {
    case "message_update": {
      const inner = event.assistantMessageEvent;
      if (inner.type === "text_delta") {
        Deno.stdout.writeSync(new TextEncoder().encode(inner.delta));
      }
      break;
    }
    case "tool_execution_start":
      console.log(`\n  🔧 ${event.toolName}(${JSON.stringify(event.args)})`);
      break;
    case "turn_end":
      for (const r of event.toolResults) {
        const text = r.content.map((c) => (c.type === "text" ? c.text : "[image]")).join(" ");
        console.log(`  ↳ ${r.toolName} → ${text}`);
      }
      break;
    case "agent_end":
      console.log(`\n✅ agent_end — transcript has ${event.messages.length} messages`);
      break;
  }
});

console.log("Agent constructed and subscribed ✔");

## 5. Run it

`agent.prompt(text)` starts a run. The Agent loops on its own — calling the model,
executing tool calls, feeding results back — until the model produces a final answer
with no more tool calls. `waitForIdle()` resolves once the run (and all event
listeners) have settled.

The prompt below needs **both** tools, so watch the agent chain them automatically.

In [ ]:
await agent.prompt("What's the weather in Rome, and what is 128 divided by 4?");
await agent.waitForIdle();
unsubscribe();

### The resulting transcript

Everything the Agent accumulated lives in `agent.state.messages` — user, assistant,
and tool-result messages, in order. We never touched it by hand.

In [ ]:
let assistantTokens = 0;
for (const msg of agent.state.messages) {
  if (msg.role === "user") {
    const text = typeof msg.content === "string"
      ? msg.content
      : msg.content.map((c) => (c.type === "text" ? c.text : "[image]")).join(" ");
    console.log(`👤 user: ${text}`);
  } else if (msg.role === "assistant") {
    for (const block of msg.content) {
      if (block.type === "text") console.log(`🤖 assistant: ${block.text}`);
      else if (block.type === "toolCall") console.log(`🤖 → tool call: ${block.name}(${JSON.stringify(block.arguments)})`);
    }
    assistantTokens += msg.usage.input + msg.usage.output;
  } else if (msg.role === "toolResult") {
    const text = msg.content.map((c) => (c.type === "text" ? c.text : "[image]")).join(" ");
    console.log(`🔧 ${msg.toolName} result: ${text}`);
  }
}
console.log(`\nTotal tokens across assistant turns: ${assistantTokens}`);

## ✅ What just happened

The `Agent` ran the **entire tool-calling loop for you**:

1. Sent your prompt to the Azure model through `streamFn`.
2. Saw the model request `get_weather` and `calculator`, and **executed them itself**.
3. Fed the tool results back and asked the model again — repeating until a final answer.
4. Emitted `message_update` / `tool_execution_start` / `turn_end` / `agent_end` events
   the whole way, and stored the full transcript in `agent.state.messages`.

### Manual loop (040) vs the `Agent` class (this notebook)

| Concern | Notebook 04 (manual) | `Agent` |
|---|---|---|
| Conversation history | you push each message | owned in `agent.state.messages` |
| Executing tools | your `for` loop calls the functions | `AgentTool.execute` called automatically |
| When to stop | your loop checks `stopReason` | Agent stops when no tool calls remain |
| Live streaming | you iterate stream events | `subscribe()` lifecycle events |
| Steering / follow-ups | build it yourself | `agent.steer()` / `agent.followUp()` queues |

### 🎉 Series complete

- **020** streaming · **030** multi-turn chat · **040** tool calling (manual loop)
- **050** structured output · **060** vision · **070** reasoning/thinking
- **080** cost, caching & robustness · **090** multi-provider · **100** the agent framework

From one LLM round-trip in notebook 010 to a self-driving, tool-using agent here.